# Initial angle and velocity

This notebook applies the semicircular polar-scatter visualization used in the paper workflow to a public synthetic example. All measurements and condition labels in the example workbook are synthetic.


In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter

DATA_FILENAME = "example_synthetic_initial_angle_velocity.xlsx"
SHEET_NAME = "Example data"

# Edit this list to plot any conditions found in the workbook.
CONDITIONS = [
    "CONTROL",
    "TREATMENT_W",
    "TREATMENT_X",
    "TREATMENT_Y",
    "TREATMENT_Z",
]

ANGLE_COLUMN = "Initial angle (degrees)"
VELOCITY_COLUMN = "Velocity (µm/min)"
VMIN, VMAX = 0.0, 0.9
RMAX = 0.9


In [ ]:
relative_dataset = Path("data") / DATA_FILENAME
candidates = [
    Path.cwd() / relative_dataset,
    Path.cwd().parent / relative_dataset,
    Path.cwd() / "public_repository" / relative_dataset,
    Path.cwd() / DATA_FILENAME,
]
DATA_FILE = next((path for path in candidates if path.exists()), None)

if DATA_FILE is None:
    checked = "\n".join(f"- {path.resolve()}" for path in candidates)
    raise FileNotFoundError(
        f"The included example dataset '{DATA_FILENAME}' was not found. Checked:\n{checked}"
    )

data = pd.read_excel(DATA_FILE, sheet_name=SHEET_NAME)
required_columns = {"Condition", ANGLE_COLUMN, VELOCITY_COLUMN}
missing_columns = sorted(required_columns.difference(data.columns))
if missing_columns:
    raise KeyError(f"Required columns not found: {missing_columns}")

missing_conditions = [condition for condition in CONDITIONS if condition not in set(data["Condition"])]
if missing_conditions:
    raise KeyError(f"Conditions not found in the workbook: {missing_conditions}")

print("Dataset:", DATA_FILE.resolve())
print(data.groupby("Condition").size().rename("n"))


In [ ]:
def safe_filename(text):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", text).strip("_").lower()


def plot_initial_angle_velocity(condition_data, condition, output_directory=None):
    angles = pd.to_numeric(condition_data[ANGLE_COLUMN], errors="coerce")
    velocities = pd.to_numeric(condition_data[VELOCITY_COLUMN], errors="coerce")
    valid = angles.notna() & velocities.notna()
    angles = angles[valid].to_numpy()
    velocities = velocities[valid].to_numpy()

    if angles.size == 0:
        raise ValueError(f"No complete angle-velocity pairs were found for {condition}.")
    if np.any((angles < 0) | (angles > 180)):
        raise ValueError(f"{condition} contains initial angles outside 0-180 degrees.")
    if np.any((velocities < VMIN) | (velocities > VMAX)):
        raise ValueError(f"{condition} contains velocities outside {VMIN}-{VMAX} µm/min.")

    fig, ax = plt.subplots(
        figsize=(10, 5),
        dpi=300,
        subplot_kw={"projection": "polar"},
        facecolor="none",
    )
    scatter = ax.scatter(
        np.deg2rad(angles),
        velocities,
        s=50,
        cmap="viridis",
        c=velocities,
        vmin=VMIN,
        vmax=VMAX,
        alpha=0.8,
    )

    ax.set_theta_zero_location("W")
    ax.set_theta_direction(-1)
    ax.set_thetamin(0)
    ax.set_thetamax(180)
    ax.set_xticks(np.deg2rad(np.arange(0, 181, 30)))
    ax.set_xticklabels([f"{degree}°" for degree in range(0, 181, 30)])
    ax.set_ylim(0, RMAX)
    radial_ticks = np.linspace(0, RMAX, 5)
    ax.set_yticks(radial_ticks)
    ax.set_yticklabels([f"{tick:.1f}" for tick in radial_ticks], fontsize=9)
    ax.set_ylabel("Velocity (µm/min)", labelpad=25)
    ax.set_title(f"Initial angle: {condition} (n = {angles.size})", pad=20)

    position = ax.get_position()
    color_axis = fig.add_axes([position.x0, position.y0 - 0.01, position.width, 0.03])
    colorbar = fig.colorbar(scatter, cax=color_axis, orientation="horizontal")
    colorbar.set_label("Velocity (µm/min)")
    colorbar.set_ticks(np.linspace(VMIN, VMAX, 5))
    colorbar.ax.xaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:.1f}"))
    colorbar.ax.tick_params(labelsize=8)

    if output_directory is not None:
        output_directory.mkdir(parents=True, exist_ok=True)
        filename = f"initial_angle_velocity_{safe_filename(condition)}.png"
        fig.savefig(output_directory / filename, dpi=300, bbox_inches="tight", transparent=True)

    return fig, ax


In [ ]:
OUTPUT_DIRECTORY = Path("outputs") / "initial_angle_velocity"

for condition in CONDITIONS:
    subset = data.loc[data["Condition"] == condition]
    figure, axis = plot_initial_angle_velocity(subset, condition, OUTPUT_DIRECTORY)
    plt.show()

print(f"PNG files were saved in: {OUTPUT_DIRECTORY.resolve()}")


## Using other conditions or data

Edit `CONDITIONS` to select which groups to plot. To use another workbook, place it in `data/` and update `DATA_FILENAME`. The workbook must contain `Condition`, `Initial angle (degrees)`, and `Velocity (µm/min)` columns. Each row must preserve the paired angle and velocity from one observation.
